# BoundaryObserver repository simulation audit and paper figures

This notebook reproduces the executable finite study used in **Boundary Limited Observer Systems**.

It is tied to repository commit:

`eb96b984ae999ee6cbc0aae33c2d501c46d9540f`

The notebook does not contain manually entered result arrays. It reads the committed CSV files generated by `simulations/run_simulations.py`. In Google Colab it clones the public repository and checks out the exact commit. When run from the paper package, it can instead use the bundled byte-identical CSV and PNG files.

The notebook performs five separate tasks:

1. runs the repository simulation runner and eighteen tests when the repository is available;
2. loads all twelve committed CSV files;
3. verifies the finite multiplicity identities from the loaded data;
4. displays the six original repository PNG figures; and
5. regenerates the six plots as vector PDF and SVG files using the repository plotting logic.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

COMMIT = "eb96b984ae999ee6cbc0aae33c2d501c46d9540f"
REPOSITORY = "https://github.com/ezernackchristopher97-cloud/BoundaryObserver.git"

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print({"commit": COMMIT, "in_colab": IN_COLAB, "python": sys.version.split()[0]})

In [ ]:
# Install only the packages required by the executable study and this notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib", "pytest"],
    check=True,
)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
# Clone and pin the repository in Colab. A local paper-package run may use bundled data instead.
REPO_ROOT = Path("/content/BoundaryObserver") if IN_COLAB else Path.cwd() / "BoundaryObserver"
repo_available = False

if IN_COLAB:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPOSITORY, str(REPO_ROOT)], check=True)
    subprocess.run(["git", "checkout", COMMIT], cwd=REPO_ROOT, check=True)
    checked = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
    assert checked == COMMIT
    repo_available = True
elif (REPO_ROOT / ".git").exists():
    checked = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
    repo_available = checked == COMMIT

print({"repository_available": repo_available, "repository_root": str(REPO_ROOT)})

In [ ]:
# Run the exact repository calculations and test suite when the pinned repository is available.
if repo_available:
    subprocess.run([sys.executable, "simulations/run_simulations.py"], cwd=REPO_ROOT, check=True)
    subprocess.run([sys.executable, "-m", "pytest", "simulations/tests", "-q"], cwd=REPO_ROOT, check=True)
else:
    print("Pinned repository not present. Using the paper package's committed outputs for data and figures.")

In [ ]:
# Resolve the data and original-figure directories without creating result values manually.
NOTEBOOK_DIR = Path.cwd()
package_candidates = [
    NOTEBOOK_DIR / "data" / "original_csv",
    NOTEBOOK_DIR.parent / "data" / "original_csv",
    Path("/content") / "data" / "original_csv",
]

if repo_available:
    DATA_DIR = REPO_ROOT / "simulations" / "results"
    ORIGINAL_PNG_DIR = DATA_DIR
else:
    DATA_DIR = next((p for p in package_candidates if p.exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Could not locate bundled data/original_csv. Upload the paper package or run in Colab.")
    ORIGINAL_PNG_DIR = DATA_DIR.parent.parent / "original_repo_pngs"

csv_names = [
    "model_summary.csv",
    "multiplicity_summary.csv",
    "experiment_outcomes.csv",
    "predicate_success_profiles.csv",
    "predicate_truth_profiles.csv",
    "empirical_quotient.csv",
    "nonidentifiable_properties.csv",
    "observation_fibers.csv",
    "interface_summary.csv",
    "refinement_summary.csv",
    "quotient_maps.csv",
    "collapsed_pairs.csv",
]

tables = {name: pd.read_csv(DATA_DIR / name) for name in csv_names}
summary = pd.DataFrame(
    [{"file": name, "rows": len(frame), "columns": len(frame.columns)} for name, frame in tables.items()]
)
display(summary)

In [ ]:
# Verify the closed-form finite multiplicity identities from the loaded committed data.
mult = tables["multiplicity_summary.csv"].copy()
m = mult["hidden_multiplicity"].astype(int)

checks = pd.DataFrame({
    "hidden_multiplicity": m,
    "latent_equals_6m": mult["latent_state_count"].astype(int).eq(6 * m),
    "reachable_equals_6": mult["reachable_observation_count"].astype(int).eq(6),
    "average_fiber_equals_m": mult["average_fiber_size"].astype(float).eq(m.astype(float)),
    "maximum_fiber_equals_m": mult["maximum_fiber_size"].astype(int).eq(m),
    "collisions_equal_3m_times_m_minus_1": mult["latent_collision_count"].astype(int).eq(3 * m * (m - 1)),
    "empirically_equivalent_to_base": mult["empirically_equivalent_to_base"].astype(bool),
})
assert checks.drop(columns="hidden_multiplicity").to_numpy().all()
display(checks)

experiments = tables["experiment_outcomes.csv"]
assert experiments["matches_base_outcomes"].astype(bool).all()

pred_success = tables["predicate_success_profiles.csv"]
pred_truth = tables["predicate_truth_profiles.csv"]
assert pred_success["matches_base_success_profile"].astype(bool).all()
assert pred_truth["matches_base_truth_profile"].astype(bool).all()

refinement = tables["refinement_summary.csv"]
assert refinement["observation_preserved"].astype(bool).all()
assert refinement["quotient_map_surjective"].astype(bool).all()

print("All loaded-data invariance and identity checks passed.")

In [ ]:
# Display the six original committed repository PNG figures.
original_names = [
    "latent_state_count_vs_hidden_multiplicity.png",
    "reachable_observation_count_vs_hidden_multiplicity.png",
    "maximum_fiber_size_vs_hidden_multiplicity.png",
    "experiment_outcome_count_vs_hidden_multiplicity.png",
    "observable_quotient_size_by_interface.png",
    "collapsed_pairs_by_refinement.png",
]

for name in original_names:
    path = ORIGINAL_PNG_DIR / name
    print(name)
    display(Image(filename=str(path)))

In [ ]:
# Regenerate the six repository plots from the committed CSV files as vector PDF and SVG.
VECTOR_DIR = Path("paper_vector_figures")
VECTOR_DIR.mkdir(exist_ok=True)

def save_vector(fig, stem):
    fig.tight_layout()
    fig.savefig(VECTOR_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(VECTOR_DIR / f"{stem}.svg", bbox_inches="tight")
    plt.show()
    plt.close(fig)

for metric, ylabel, title, stem in [
    ("latent_state_count", "Latent state count", "Latent states grow with hidden multiplicity", "latent_state_count_vs_hidden_multiplicity"),
    ("reachable_observation_count", "Reachable observation count", "Reachable observations remain fixed", "reachable_observation_count_vs_hidden_multiplicity"),
    ("maximum_fiber_size", "Maximum fiber size", "Maximum fiber size grows with hidden multiplicity", "maximum_fiber_size_vs_hidden_multiplicity"),
]:
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.plot(mult["hidden_multiplicity"], mult[metric], marker="o", linewidth=2)
    ax.set_xlabel("Hidden multiplicity")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(mult["hidden_multiplicity"])
    ax.grid(alpha=0.3)
    save_vector(fig, stem)

model_names = {
    "injective_base_1_fold_extension",
    "injective_base_2_fold_extension",
    "injective_base_3_fold_extension",
    "injective_base_5_fold_extension",
    "injective_base_8_fold_extension",
}
experiment_order = ["identity", "threshold_at_3", "categorical_partition", "compressed_width_2"]
styles = (
    {"marker": "o", "linewidth": 2.2, "linestyle": "-"},
    {"marker": "s", "linewidth": 2.2, "linestyle": "--"},
    {"marker": "^", "linewidth": 5.0, "linestyle": "-"},
    {"marker": "D", "linewidth": 2.0, "linestyle": ":"},
)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for name, style in zip(experiment_order, styles):
    selected = experiments[(experiments["experiment"] == name) & experiments["model"].isin(model_names)].sort_values("hidden_multiplicity")
    ax.plot(selected["hidden_multiplicity"], selected["outcome_count"], label=name, **style)
ax.set_xlabel("Hidden multiplicity")
ax.set_ylabel("Possible experiment outcome count")
ax.set_title("Observation-only outcomes remain fixed")
ax.set_xticks(sorted(mult["hidden_multiplicity"].unique()))
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
save_vector(fig, "experiment_outcome_count_vs_hidden_multiplicity")

interfaces = tables["interface_summary.csv"]
labels = interfaces["interface"].str.replace("_interface", "", regex=False)
sizes = interfaces["observable_quotient_size"].astype(int)
fig, ax = plt.subplots(figsize=(8.0, 4.8))
bars = ax.bar(labels, sizes, color=("#2563eb", "#7c3aed", "#db2777", "#0891b2"))
ax.bar_label(bars, padding=3)
ax.set_ylabel("Observable quotient class count")
ax.set_title("Finite interfaces retain different numbers of distinctions")
ax.set_ylim(0, sizes.max() + 1.5)
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=15)
save_vector(fig, "observable_quotient_size_by_interface")

labels = refinement["refinement"].str.replace("_to_", " → ", regex=False)
counts = refinement["collapsed_world_pair_count"].astype(int)
fig, ax = plt.subplots(figsize=(8.0, 4.8))
bars = ax.barh(labels, counts, color="#c2410c")
ax.bar_label(bars, padding=3)
ax.set_xlabel("Pairs distinguished by fine and identified by coarse")
ax.set_title("Collapsed pairs under finite coarsening")
ax.set_xlim(0, counts.max() + 2)
ax.grid(axis="x", alpha=0.25)
save_vector(fig, "collapsed_pairs_by_refinement")

In [ ]:
# Record byte-level provenance for every loaded CSV and original PNG.
def digest(path):
    raw = path.read_bytes()
    return {"sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw)}

manifest = {
    "repository": REPOSITORY,
    "commit": COMMIT,
    "csv_files": {name: digest(DATA_DIR / name) for name in csv_names},
    "original_png_files": {name: digest(ORIGINAL_PNG_DIR / name) for name in original_names},
    "vector_outputs": sorted(path.name for path in VECTOR_DIR.iterdir()),
}
Path("notebook_provenance_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2)[:5000])

In [ ]:
# Package the vector outputs and provenance record.
archive = shutil.make_archive("BoundaryObserver_Colab_Vector_Figures", "zip", VECTOR_DIR)
print(archive)
if IN_COLAB:
    files.download(archive)